<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [7]</a>'.</span>

# Template 05: Model Training

**Purpose:** Train XGBoost model and generate predictions

**Inputs:**
- data/04_train.parquet
- data/04_test.parquet

**Outputs:**
- models/xgb_model.json
- results/05_metrics.yaml
- results/05_predictions.parquet

In [1]:
config_path = "config/car_coll/v1"

In [2]:
# Parameters
config_path = "config/car_coll/v1"


In [3]:
import pandas as pd
import numpy as np
import xgboost as xgb
import yaml
import os
import sys
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error

sys.path.insert(0, str(Path.cwd() / 'lib'))
from utils import setup_notebook_environment

print("########################################")
print("# STAGE 05: MODEL TRAINING")
print("########################################")

project_root = setup_notebook_environment()

########################################
# STAGE 05: MODEL TRAINING
########################################


In [4]:
config_file = f"{config_path}/config.yaml"
with open(config_file, 'r') as f:
    cfg = yaml.safe_load(f)

output_base = cfg['paths']['output_base']
target = cfg['experiment']['target']

In [5]:
# Load train/test data
train = pd.read_parquet(f"{output_base}/data/04_train.parquet")
test = pd.read_parquet(f"{output_base}/data/04_test.parquet")

print(f"\n* Train: {train.shape}")
print(f"* Test: {test.shape}")


* Train: (2716120, 109)
* Test: (2707950, 109)


In [6]:
# Load feature list
features_df = pd.read_csv(f"{config_path}/{cfg['features']['inclusion_file']}", comment='#')
feature_cols = features_df['column_name'].tolist()

# Filter to available features
available_features = [f for f in feature_cols if f in train.columns]
print(f"\n* Features: {len(available_features)}/{len(feature_cols)} available")

X_train = train[available_features]
y_train = train[target]
X_test = test[available_features]
y_test = test[target]


* Features: 90/90 available


<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [7]:
# Train XGBoost model
print(f"\n* Training XGBoost...")

xgb_params = cfg['xgboost'].copy()
n_estimators = xgb_params.pop('n_estimators', 5000)

model = xgb.XGBRegressor(n_estimators=n_estimators, **xgb_params)
model.fit(X_train, y_train, verbose=100)

print(f"\n* Model trained")


* Training XGBoost...


ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:NumMajViol_raw: object, NumMajinAcc_raw: object, NumMinAcc_raw: object, NumMinViol_raw: object, NumSpdViol_raw: object, late_payments_raw: object

In [ ]:
# Predictions
train['pred'] = model.predict(X_train)
test['pred'] = model.predict(X_test)

# Metrics
train_mae = mean_absolute_error(y_train, train['pred'])
test_mae = mean_absolute_error(y_test, test['pred'])
train_rmse = np.sqrt(mean_squared_error(y_train, train['pred']))
test_rmse = np.sqrt(mean_squared_error(y_test, test['pred']))

print(f"\n* Metrics:")
print(f"  Train MAE: {train_mae:.4f}")
print(f"  Test MAE: {test_mae:.4f}")
print(f"  Train RMSE: {train_rmse:.4f}")
print(f"  Test RMSE: {test_rmse:.4f}")

In [ ]:
# Save model
model_file = f"{output_base}/models/xgb_model.json"
os.makedirs(f"{output_base}/models", exist_ok=True)
model.save_model(model_file)
print(f"\n* Model saved: {model_file}")

In [ ]:
# Save metrics
metrics = {
    'train_mae': float(train_mae),
    'test_mae': float(test_mae),
    'train_rmse': float(train_rmse),
    'test_rmse': float(test_rmse),
    'n_features': len(available_features),
    'train_size': len(train),
    'test_size': len(test)
}

metrics_file = f"{output_base}/results/05_metrics.yaml"
with open(metrics_file, 'w') as f:
    yaml.dump(metrics, f)

print(f"* Metrics saved: {metrics_file}")

In [ ]:
# Save predictions
pred_file = f"{output_base}/results/05_predictions.parquet"
test[['pred', target]].to_parquet(pred_file)
print(f"* Predictions saved: {pred_file}")

## Lift Charts

In [ ]:
# Prepare data for lift charts
# Assumes exposure/weight column exists - adjust as needed
weight_col = 'vin'  # Replace with actual weight column if available

# Create incurred/denom columns for lift chart
train['incurred_act'] = train[target]
train['incurred_pred'] = train['pred']
train['denom'] = 1  # Adjust if using exposures

test['incurred_act'] = test[target]
test['incurred_pred'] = test['pred']
test['denom'] = 1  # Adjust if using exposures

print(f"\n* Prepared data for lift charts")

In [ ]:
# Load lift chart function from lib
import matplotlib.pyplot as plt
import importlib.util

# Load gbm_functions module
spec = importlib.util.spec_from_file_location("gbm_functions", "lib/gbm_functions.ipynb")
# Note: This is a notebook, we'll use simpler inline version

# Simple lift chart function (inline version)
def create_lift_chart(data, weight_name, bins=10, title="Lift Chart"):
    df = data.copy()
    w = df[weight_name]
    wsum = w.sum()
    
    # Create deciles
    df = df.sort_values('pred').reset_index(drop=True)
    cum_w = w.cumsum() / wsum
    df['decile'] = np.ceil(cum_w * bins).astype(int).clip(1, bins)
    
    # Aggregate
    x = df.groupby('decile').agg({
        weight_name: 'sum',
        'incurred_act': 'sum',
        'incurred_pred': 'sum',
        'denom': 'sum'
    }).reset_index()
    
    # Calculate act/pred values
    x['act'] = x['incurred_act'] / x['denom']
    x['pred'] = x['incurred_pred'] / x['denom']
    
    # Relativities
    overall_pred = df['incurred_pred'].sum() / df['denom'].sum()
    x['act_rel'] = x['act'] / overall_pred
    x['pred_rel'] = x['pred'] / overall_pred
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(x['decile'], x['act_rel'], marker='o', label='Actual Relativity', linewidth=2)
    ax.plot(x['decile'], x['pred_rel'], marker='s', label='Predicted Relativity', linewidth=2)
    ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Decile')
    ax.set_ylabel('Relativity')
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    return fig, x

print("* Lift chart function loaded")

In [ ]:
# Train lift chart
print("\n* Generating train lift chart...")
fig_train, table_train = create_lift_chart(train, weight_col, bins=10, title="Train Lift Chart")

train_chart_file = f"{output_base}/results/05_lift_chart_train.png"
fig_train.savefig(train_chart_file, dpi=150, bbox_inches='tight')
plt.close(fig_train)

print(f"  Saved: {train_chart_file}")
print("\nTrain decile table:")
print(table_train[['decile', 'act', 'pred', 'act_rel', 'pred_rel']])

In [ ]:
# Test lift chart
print("\n* Generating test lift chart...")
fig_test, table_test = create_lift_chart(test, weight_col, bins=10, title="Test Lift Chart")

test_chart_file = f"{output_base}/results/05_lift_chart_test.png"
fig_test.savefig(test_chart_file, dpi=150, bbox_inches='tight')
plt.close(fig_test)

print(f"  Saved: {test_chart_file}")
print("\nTest decile table:")
print(table_test[['decile', 'act', 'pred', 'act_rel', 'pred_rel']])

In [ ]:
print("\n########################################")
print("# STAGE 05: COMPLETE")
print("########################################")